# 03. Tiny decoder LoRA overfit check

Prove that the loss, image tokens, gradient flow, adapter saving and reloading work before a
larger experiment. Use only reviewed smoke pages, never validation/test pages. This intentionally
fits the same page repeatedly; its accuracy is a debugging result, not evidence of generalization.

Default: decoder q/v, last eight blocks, rank 4, batch one, one 448 px tile, 40 passes.
Restart the kernel before running this notebook if the original association model is still loaded.
Use a new run name to repeat; existing experiment folders are protected against overwriting.


In [ ]:
from pathlib import Path
import sys
import json
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "models.json").exists():
    ROOT = ROOT.parent
if not (ROOT / "models.json").exists():
    raise FileNotFoundError("Start the kernel in 1pageTest or its 6_lora_training folder.")
sys.path.insert(0, str(ROOT / "6_lora_training"))
import lora_helpers as H
print("Interpreter:", sys.executable)
print("Dataset:", H.PAGE_DIR)
print("Local base model:", H.MODEL_DIR)


In [ ]:
pages = H.read_pages()
smoke_pages = H.partition(pages, "smoke")
SMOKE = H.experiment("smoke_decoder_qv_last8_r4", decoder="qv", last_layers=8,
                     rank=4, epochs=40, lr=2e-4, accumulation=1)
directory = H.train_run(SMOKE, smoke_pages, smoke=True)
display(json.loads((directory / "history.json").read_text()))


In [ ]:
import matplotlib.pyplot as plt
history = json.loads((directory / "history.json").read_text())
plt.plot([r["epoch"] for r in history], [r["train_loss"] for r in history])
plt.xlabel("Pass over smoke pages")
plt.ylabel("Assistant-token training loss")
plt.show()
result = H.evaluate_run(directory / "best", smoke_pages,
                        destination=directory / "smoke_predictions.json")
display(result["summary"])


Expected checks: only language_model adapter parameters are trainable; first-step gradient
norm is finite and nonzero; loss declines; reloaded predictions use legal IDs. If these fail,
inspect human labels, token masking and the reported gradients before broadening training.
A zero gradient in LoRA A at initialization alone is normal; the whole adapter group must
receive a gradient. A single low-resolution page may memorize labels rather than use the image.
